# 03. Model and Cost Comparison

Goal: compare accuracy and runtime trade-offs between a fast model (flash) and a more capable model (pro).

## Step 1: Install Dependencies

In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git
%pip install -q -r esmt-workshop/requirements.txt


## Step 2: Load Shared Utilities

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/esmt-workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses, process_single_address
print('PROJECT_ROOT =', PROJECT_ROOT)


## Step 3: Student Identity Parameter

In [ ]:
from google.colab import auth

auth.authenticate_user()

gcloud_email = !gcloud config get-value account
print(f"Authenticated as: {gcloud_email[0] if gcloud_email else 'Unknown'}")
os.environ['WORKSHOP_EMAIL'] = gcloud_email[0] if gcloud_email else os.environ.get('WORKSHOP_EMAIL','')

# Students only provide email; proxy endpoint details are managed by organizers.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', 'student@example.com')
print('STUDENT_EMAIL =', STUDENT_EMAIL)


## Step 4: Run Two-Model Comparison

In [ ]:
import time

# Prompt is editable directly in the notebook (no external prompt files).
USE_CUSTOM_PROMPT = True
PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Town Name should be city/locality only.
- Postal Code should include only postal token(s).
- Country Code must be ISO alpha-2 uppercase.
- Use empty strings when unknown.
'''

custom_prompt = PROMPT_TEMPLATE if USE_CUSTOM_PROMPT else None

# Load labeled dev set for scoring.
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')

# Model choices
FLASH_MODEL = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
PRO_MODEL = os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro')

# Shared generation params (keep consistent for fair comparison)
TEMPERATURE = 0.0
TOP_P = 0.95
TOP_K = 40
MAX_TOKENS = None  # keep toolkit default unless you need to cap
MAX_WORKERS = 8

print('Running Flash model:', FLASH_MODEL)
t0 = time.perf_counter()
pred_flash = process_batch_addresses(
    dev_df,
    email=STUDENT_EMAIL,
    stage='model_and_cost',
    model=FLASH_MODEL,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    custom_prompt_template=custom_prompt,
    max_workers=MAX_WORKERS,
)
runtime_flash = time.perf_counter() - t0
report_flash = evaluate_predictions(pred_flash, dev_df, email=STUDENT_EMAIL)

print('\nRunning Pro model:', PRO_MODEL)
t0 = time.perf_counter()
pred_pro = process_batch_addresses(
    dev_df,
    email=STUDENT_EMAIL,
    stage='model_and_cost',
    model=PRO_MODEL,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    custom_prompt_template=custom_prompt,
    max_workers=MAX_WORKERS,
)
runtime_pro = time.perf_counter() - t0
report_pro = evaluate_predictions(pred_pro, dev_df, email=STUDENT_EMAIL)

print('\nFlash summary:', report_flash.get('summary', {}))
print('Pro summary:', report_pro.get('summary', {}))

# Side-by-side comparison table
comparison_df = pd.DataFrame([
    {'model': FLASH_MODEL, 'micro_accuracy': report_flash.get('summary', {}).get('micro_accuracy', 0.0), 'runtime_sec': runtime_flash},
    {'model': PRO_MODEL, 'micro_accuracy': report_pro.get('summary', {}).get('micro_accuracy', 0.0), 'runtime_sec': runtime_pro},
])

comparison_df


## Step 5: Error Analysis

In [ ]:
# Optional: error analysis (where did the models differ?)

print('Flash field_metrics:')
display(report_flash.get('field_metrics', pd.DataFrame()))

print('\nPro field_metrics:')
display(report_pro.get('field_metrics', pd.DataFrame()))

print('\nSample Flash mismatches:')
display(report_flash.get('mismatches', pd.DataFrame()).head(10))

print('\nSample Pro mismatches:')
display(report_pro.get('mismatches', pd.DataFrame()).head(10))


## Step 6: Save Model-Comparison Artifacts

In [ ]:
# Save predictions and evaluation files for stage comparison (aligned with original notebook naming).
out_dir = PROJECT_ROOT / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

flash_pred_path = out_dir / '03_model_and_cost_flash_predictions.csv'
pro_pred_path = out_dir / '03_model_and_cost_pro_predictions.csv'

flash_report_dir = out_dir / 'report_03_model_and_cost_flash'
pro_report_dir = out_dir / 'report_03_model_and_cost_pro'

pred_flash.to_csv(flash_pred_path, index=False)
pred_pro.to_csv(pro_pred_path, index=False)

save_evaluation_report(report_flash, flash_report_dir)
save_evaluation_report(report_pro, pro_report_dir)

print('Saved Flash predictions:', flash_pred_path)
print('Saved Flash report:', flash_report_dir)
print('Saved Pro predictions:', pro_pred_path)
print('Saved Pro report:', pro_report_dir)


## Discussion

- When would you choose the flash model over the pro model?
- If pro is slightly better, is the extra latency/cost worth it?
- What would you measure in a production system besides accuracy and runtime?